In [3]:
import sys
import os

# Thêm thư mục gốc của dự án vào đường dẫn Python
# Sử dụng os.getcwd() vì __file__ không tồn tại trong Notebook
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from models.account import Account
from models.customer import Customer

# print("Đã nhập thành công các mô-dun!")

Đã nhập thành công các mô-dun!


In [ ]:
# Hệ thống bảo mật ngân hàng siêu phức tạp
# Mô phỏng các tính năng bảo mật của ngân hàng lớn nhất thế giới

import hashlib
import secrets
import time
import json
import base64
import hmac
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Tuple
import re
from functools import wraps
import threading
import queue
import random
import string
import uuid

# 1. Hệ thống mã hóa đa lớp
class MultiLayerEncryption:
    """Mã hóa đa lớp với AES-256 và RSA"""
    
    def __init__(self):
        self.master_key = secrets.token_bytes(32)
        self.iv = secrets.token_bytes(16)
        
    def _xor_encrypt(self, data: bytes, key: bytes) -> bytes:
        """XOR encryption"""
        return bytes(a ^ b for a, b in zip(data, key * (len(data) // len(key) + 1)))
    
    def encrypt_sensitive_data(self, data: str) -> str:
        """Mã hóa dữ liệu nhạy cảm với nhiều lớp"""
        # Lớp 1: Base64 encoding
        encoded = base64.b64encode(data.encode())
        
        # Lớp 2: XOR với master key
        encrypted = self._xor_encrypt(encoded, self.master_key)
        
        # Lớp 3: Thêm salt và hash
        salt = secrets.token_hex(16)
        final = salt + ":" + base64.b64encode(encrypted).decode()
        
        return final
    
    def decrypt_sensitive_data(self, encrypted_data: str) -> str:
        """Giải mã dữ liệu"""
        try:
            salt, data = encrypted_data.split(":", 1)
            encrypted = base64.b64decode(data)
            decrypted = self._xor_encrypt(encrypted, self.master_key)
            decoded = base64.b64decode(decrypted).decode()
            return decoded
        except Exception as e:
            raise Exception(f"Decryption failed: {str(e)}")


In [ ]:
# 2. Hệ thống xác thực đa yếu tố (MFA)
class MultiFactorAuthentication:
    """Xác thực đa yếu tố với TOTP, SMS, Biometric"""
    
    def __init__(self):
        self.totp_secrets = {}
        self.sms_codes = {}
        self.biometric_data = {}
        self.device_fingerprints = {}
        
    def generate_totp_secret(self, user_id: str) -> str:
        """Tạo secret key cho TOTP"""
        secret = base64.b32encode(secrets.token_bytes(20)).decode('utf-8')
        self.totp_secrets[user_id] = secret
        return secret
    
    def generate_totp_code(self, secret: str) -> str:
        """Tạo mã TOTP 6 số"""
        # Mô phỏng TOTP algorithm
        timestamp = int(time.time() // 30)
        hmac_hash = hmac.new(
            base64.b32decode(secret),
            timestamp.to_bytes(8, byteorder='big'),
            hashlib.sha1
        ).digest()
        
        offset = hmac_hash[-1] & 0x0f
        code = hmac_hash[offset:offset+4]
        code_int = int.from_bytes(code, byteorder='big') & 0x7fffffff
        code_str = str(code_int % 1000000).zfill(6)
        
        return code_str
    
    def verify_totp(self, user_id: str, token: str) -> bool:
        """Xác thực TOTP token"""
        if user_id not in self.totp_secrets:
            return False
        
        secret = self.totp_secrets[user_id]
        expected_code = self.generate_totp_code(secret)
        
        # Cho phép 1 window trước và sau
        for window in [-1, 0, 1]:
            timestamp = int(time.time() // 30) + window
            hmac_hash = hmac.new(
                base64.b32decode(secret),
                timestamp.to_bytes(8, byteorder='big'),
                hashlib.sha1
            ).digest()
            
            offset = hmac_hash[-1] & 0x0f
            code = hmac_hash[offset:offset+4]
            code_int = int.from_bytes(code, byteorder='big') & 0x7fffffff
            code_str = str(code_int % 1000000).zfill(6)
            
            if code_str == token:
                return True
        
        return False
    
    def send_sms_code(self, user_id: str, phone: str) -> str:
        """Gửi mã SMS (mô phỏng)"""
        code = ''.join(random.choices(string.digits, k=6))
        self.sms_codes[user_id] = {
            'code': code,
            'timestamp': time.time(),
            'phone': phone,
            'attempts': 0
        }
        print(f"[SMS] Đang gửi mã {code} đến {phone}")
        return code
    
    def verify_sms_code(self, user_id: str, code: str) -> bool:
        """Xác thực mã SMS"""
        if user_id not in self.sms_codes:
            return False
        
        stored = self.sms_codes[user_id]
        
        # Kiểm tra số lần thử
        if stored['attempts'] >= 3:
            del self.sms_codes[user_id]
            return False
        
        stored['attempts'] += 1
        
        # Mã hết hạn sau 5 phút
        if time.time() - stored['timestamp'] > 300:
            del self.sms_codes[user_id]
            return False
        
        if stored['code'] == code:
            del self.sms_codes[user_id]
            return True
        
        return False
    
    def register_biometric(self, user_id: str, biometric_data: str):
        """Đăng ký dữ liệu sinh trắc học"""
        # Hash biometric data với salt
        salt = secrets.token_hex(16)
        biometric_hash = hashlib.pbkdf2_hmac(
            'sha256',
            biometric_data.encode(),
            salt.encode(),
            100000
        ).hex()
        
        self.biometric_data[user_id] = {
            'hash': biometric_hash,
            'salt': salt,
            'registered_at': datetime.now().isoformat(),
            'last_used': None
        }
    
    def verify_biometric(self, user_id: str, biometric_data: str) -> bool:
        """Xác thực sinh trắc học"""
        if user_id not in self.biometric_data:
            return False
        
        stored = self.biometric_data[user_id]
        biometric_hash = hashlib.pbkdf2_hmac(
            'sha256',
            biometric_data.encode(),
            stored['salt'].encode(),
            100000
        ).hex()
        
        if stored['hash'] == biometric_hash:
            stored['last_used'] = datetime.now().isoformat()
            return True
        
        return False
    
    def register_device(self, user_id: str, device_info: Dict) -> str:
        """Đăng ký thiết bị tin cậy"""
        device_id = str(uuid.uuid4())
        fingerprint = hashlib.sha256(
            json.dumps(device_info, sort_keys=True).encode()
        ).hexdigest()
        
        if user_id not in self.device_fingerprints:
            self.device_fingerprints[user_id] = {}
        
        self.device_fingerprints[user_id][device_id] = {
            'fingerprint': fingerprint,
            'info': device_info,
            'registered_at': datetime.now().isoformat(),
            'last_seen': datetime.now().isoformat()
        }
        
        return device_id
    
    def verify_device(self, user_id: str, device_id: str, device_info: Dict) -> bool:
        """Xác thực thiết bị"""
        if user_id not in self.device_fingerprints:
            return False
        
        if device_id not in self.device_fingerprints[user_id]:
            return False
        
        stored = self.device_fingerprints[user_id][device_id]
        fingerprint = hashlib.sha256(
            json.dumps(device_info, sort_keys=True).encode()
        ).hexdigest()
        
        if stored['fingerprint'] == fingerprint:
            stored['last_seen'] = datetime.now().isoformat()
            return True
        
        return False


In [ ]:
# 3. Hệ thống phát hiện gian lận và bất thường
class FraudDetectionSystem:
    """Hệ thống AI phát hiện gian lận thời gian thực"""
    
    def __init__(self):
        self.transaction_history = {}
        self.risk_scores = {}
        self.blocked_ips = set()
        self.suspicious_patterns = []
        self.ml_model_weights = self._initialize_ml_weights()
        
    def _initialize_ml_weights(self) -> Dict:
        """Khởi tạo trọng số cho mô hình ML"""
        return {
            'amount_threshold': 0.3,
            'frequency_threshold': 0.25,
            'location_threshold': 0.2,
            'time_threshold': 0.15,
            'device_threshold': 0.1
        }
    
    def analyze_transaction(self, transaction: Dict) -> Tuple[float, List[str]]:
        """Phân tích giao dịch và tính điểm rủi ro"""
        user_id = transaction['user_id']
        risk_score = 0.0
        risk_factors = []
        
        # 1. Kiểm tra số tiền bất thường
        if transaction['amount'] > 10000000:  # 10 triệu VND
            risk_score += 0.3
            risk_factors.append("Số tiền giao dịch lớn")
        
        # 2. Kiểm tra tần suất giao dịch
        if user_id in self.transaction_history:
            recent_txns = [
                txn for txn in self.transaction_history[user_id]
                if time.time() - txn['timestamp'] < 3600  # 1 giờ
            ]
            if len(recent_txns) > 5:
                risk_score += 0.25
                risk_factors.append("Tần suất giao dịch cao")
        
        # 3. Kiểm tra vị trí địa lý
        if 'location' in transaction:
            if self._is_suspicious_location(user_id, transaction['location']):
                risk_score += 0.2
                risk_factors.append("Vị trí đáng ngờ")
        
        # 4. Kiểm tra thời gian giao dịch
        hour = datetime.fromtimestamp(transaction['timestamp']).hour
        if hour < 6 or hour > 23:
            risk_score += 0.15
            risk_factors.append("Giao dịch ngoài giờ")
        
        # 5. Kiểm tra thiết bị
        if 'device_id' in transaction:
            if self._is_new_device(user_id, transaction['device_id']):
                risk_score += 0.1
                risk_factors.append("Thiết bị mới")
        
        # Lưu lịch sử
        if user_id not in self.transaction_history:
            self.transaction_history[user_id] = []
        self.transaction_history[user_id].append(transaction)
        
        # Giới hạn lịch sử
        if len(self.transaction_history[user_id]) > 1000:
            self.transaction_history[user_id] = self.transaction_history[user_id][-500:]
        
        return risk_score, risk_factors
    
    def _is_suspicious_location(self, user_id: str, location: Dict) -> bool:
        """Kiểm tra vị trí đáng ngờ"""
        if user_id not in self.transaction_history:
            return False
        
        # Lấy vị trí giao dịch gần nhất
        recent_locations = [
            txn.get('location') for txn in self.transaction_history[user_id][-10:]
            if 'location' in txn
        ]
        
        if not recent_locations:
            return False
        
        # Kiểm tra khoảng cách (giả lập)
        for recent_loc in recent_locations:
            distance = abs(location['lat'] - recent_loc['lat']) + abs(location['lon'] - recent_loc['lon'])
            if distance > 10:  # Ngưỡng khoảng cách
                return True
        
        return False
    
    def _is_new_device(self, user_id: str, device_id: str) -> bool:
        """Kiểm tra thiết bị mới"""
        if user_id not in self.transaction_history:
            return True
        
        known_devices = set(
            txn.get('device_id') for txn in self.transaction_history[user_id]
            if 'device_id' in txn
        )
        
        return device_id not in known_devices
    
    def block_ip(self, ip_address: str, reason: str):
        """Chặn IP đáng ngờ"""
        self.blocked_ips.add(ip_address)
        print(f"[SECURITY] Blocked IP {ip_address}: {reason}")
    
    def is_ip_blocked(self, ip_address: str) -> bool:
        """Kiểm tra IP có bị chặn không"""
        return ip_address in self.blocked_ips
    
    def detect_pattern(self, transactions: List[Dict]) -> List[str]:
        """Phát hiện mẫu gian lận"""
        patterns = []
        
        # Phát hiện velocity attack
        if len(transactions) > 10:
            time_diffs = [
                transactions[i]['timestamp'] - transactions[i-1]['timestamp']
                for i in range(1, len(transactions))
            ]
            avg_diff = sum(time_diffs) / len(time_diffs)
            if avg_diff < 5:  # Dưới 5 giây giữa các giao dịch
                patterns.append("Velocity attack detected")
        
        # Phát hiện card testing
        amounts = [txn['amount'] for txn in transactions]
        if len(set(amounts)) == 1 and amounts[0] < 10000:  # Cùng số tiền nhỏ
            patterns.append("Card testing pattern detected")
        
        return patterns


In [ ]:
# 4. Hệ thống quản lý phiên và JWT
class SessionManager:
    """Quản lý phiên làm việc an toàn với JWT"""
    
    def __init__(self):
        self.secret_key = secrets.token_urlsafe(32)
        self.sessions = {}
        self.refresh_tokens = {}
        self.blacklisted_tokens = set()
        
    def create_jwt_token(self, user_id: str, device_id: str, role: str = 'user') -> str:
        """Tạo JWT token"""
        payload = {
            'user_id': user_id,
            'device_id': device_id,
            'role': role,
            'iat': datetime.utcnow().timestamp(),
            'exp': (datetime.utcnow() + timedelta(minutes=15)).timestamp(),
            'jti': str(uuid.uuid4())  # JWT ID để có thể thu hồi
        }
        
        # Mô phỏng JWT encoding
        header = base64.urlsafe_b64encode(
            json.dumps({'alg': 'HS256', 'typ': 'JWT'}).encode()
        ).decode().rstrip('=')
        
        payload_encoded = base64.urlsafe_b64encode(
            json.dumps(payload).encode()
        ).decode().rstrip('=')
        
        signature = base64.urlsafe_b64encode(
            hmac.new(
                self.secret_key.encode(),
                f"{header}.{payload_encoded}".encode(),
                hashlib.sha256
            ).digest()
        ).decode().rstrip('=')
        
        token = f"{header}.{payload_encoded}.{signature}"
        
        # Lưu session
        self.sessions[payload['jti']] = {
            'user_id': user_id,
            'device_id': device_id,
            'created_at': datetime.utcnow(),
            'last_activity': datetime.utcnow(),
            'ip_address': None
        }
        
        return token
    
    def create_refresh_token(self, user_id: str) -> str:
        """Tạo refresh token"""
        refresh_token = secrets.token_urlsafe(64)
        self.refresh_tokens[refresh_token] = {
            'user_id': user_id,
            'created_at': datetime.utcnow(),
            'expires_at': datetime.utcnow() + timedelta(days=30)
        }
        return refresh_token
    
    def verify_jwt_token(self, token: str) -> Optional[Dict]:
        """Xác thực JWT token"""
        try:
            parts = token.split('.')
            if len(parts) != 3:
                return None
            
            header, payload_encoded, signature = parts
            
            # Verify signature
            expected_signature = base64.urlsafe_b64encode(
                hmac.new(
                    self.secret_key.encode(),
                    f"{header}.{payload_encoded}".encode(),
                    hashlib.sha256
                ).digest()
            ).decode().rstrip('=')
            
            if signature != expected_signature:
                return None
            
            # Decode payload
            payload = json.loads(
                base64.urlsafe_b64decode(payload_encoded + '==')
            )
            
            # Check expiration
            if payload['exp'] < datetime.utcnow().timestamp():
                return None
            
            # Check if blacklisted
            if payload['jti'] in self.blacklisted_tokens:
                return None
            
            # Update last activity
            if payload['jti'] in self.sessions:
                self.sessions[payload['jti']]['last_activity'] = datetime.utcnow()
            
            return payload
            
        except Exception:
            return None
    
    def revoke_token(self, jti: str):
        """Thu hồi token"""
        self.blacklisted_tokens.add(jti)
        if jti in self.sessions:
            del self.sessions[jti]
    
    def cleanup_expired_sessions(self):
        """Dọn dẹp phiên hết hạn"""
        now = datetime.utcnow()
        
        # Cleanup sessions
        expired_sessions = [
            jti for jti, session in self.sessions.items()
            if now - session['last_activity'] > timedelta(hours=1)
        ]
        for jti in expired_sessions:
            del self.sessions[jti]
        
        # Cleanup refresh tokens
        expired_refresh = [
            token for token, data in self.refresh_tokens.items()
            if data['expires_at'] < now
        ]
        for token in expired_refresh:
            del self.refresh_tokens[token]


In [ ]:
# 5. Hệ thống Rate Limiting và DDoS Protection
class RateLimiter:
    """Giới hạn tốc độ và bảo vệ DDoS"""
    
    def __init__(self):
        self.request_history = {}
        self.blocked_users = {}
        self.global_rate_limit = 1000  # requests per minute
        self.user_rate_limit = 100  # requests per minute per user
        self.ip_rate_limit = 200  # requests per minute per IP
        
    def check_rate_limit(self, user_id: str, ip_address: str) -> Tuple[bool, str]:
        """Kiểm tra giới hạn tốc độ"""
        current_time = time.time()
        minute_ago = current_time - 60
        
        # Kiểm tra user bị block
        if user_id in self.blocked_users:
            if current_time < self.blocked_users[user_id]['until']:
                return False, f"User blocked until {self.blocked_users[user_id]['until']}"
            else:
                del self.blocked_users[user_id]
        
        # Khởi tạo lịch sử nếu chưa có
        if user_id not in self.request_history:
            self.request_history[user_id] = {
                'requests': [],
                'ip_requests': {}
            }
        
        user_history = self.request_history[user_id]
        
        # Lọc requests trong 1 phút qua
        user_history['requests'] = [
            req for req in user_history['requests']
            if req > minute_ago
        ]
        
        # Kiểm tra user rate limit
        if len(user_history['requests']) >= self.user_rate_limit:
            self._block_user(user_id, 300)  # Block 5 phút
            return False, "User rate limit exceeded"
        
        # Kiểm tra IP rate limit
        if ip_address not in user_history['ip_requests']:
            user_history['ip_requests'][ip_address] = []
        
        user_history['ip_requests'][ip_address] = [
            req for req in user_history['ip_requests'][ip_address]
            if req > minute_ago
        ]
        
        if len(user_history['ip_requests'][ip_address]) >= self.ip_rate_limit:
            return False, "IP rate limit exceeded"
        
        # Thêm request mới
        user_history['requests'].append(current_time)
        user_history['ip_requests'][ip_address].append(current_time)
        
        return True, "OK"
    
    def _block_user(self, user_id: str, duration: int):
        """Block user trong một khoảng thời gian"""
        self.blocked_users[user_id] = {
            'until': time.time() + duration,
            'reason': 'Rate limit exceeded'
        }
    
    def detect_ddos_pattern(self, requests: List[Dict]) -> bool:
        """Phát hiện mẫu tấn công DDoS"""
        if len(requests) < 100:
            return False
        
        # Phân tích thời gian giữa các requests
        time_diffs = []
        for i in range(1, len(requests)):
            diff = requests[i]['timestamp'] - requests[i-1]['timestamp']
            time_diffs.append(diff)
        
        avg_diff = sum(time_diffs) / len(time_diffs)
        
        # Nếu requests quá đều đặn -> có thể là bot
        if avg_diff < 0.1:  # Dưới 100ms
            return True
        
        # Kiểm tra số lượng IP độc nhất
        unique_ips = set(req['ip'] for req in requests)
        if len(unique_ips) < len(requests) * 0.1:  # Ít hơn 10% IP độc nhất
            return True
        
        return False

# 6. Hệ thống Audit và Logging
class AuditLogger:
    """Ghi log toàn diện cho mọi hoạt động"""
    
    def __init__(self):
        self.audit_logs = []
        self.security_events = []
        self.compliance_logs = []
        
    def log_transaction(self, transaction: Dict):
        """Ghi log giao dịch"""
        log_entry = {
            'id': str(uuid.uuid4()),
            'timestamp': datetime.utcnow().isoformat(),
            'type': 'TRANSACTION',
            'user_id': transaction.get('user_id'),
            'amount': transaction.get('amount'),
            'from_account': transaction.get('from_account'),
            'to_account': transaction.get('to_account'),
            'ip_address': transaction.get('ip_address'),
            'device_id': transaction.get('device_id'),
            'status': transaction.get('status'),
            'risk_score': transaction.get('risk_score'),
            'hash': self._calculate_hash(transaction)
        }
        
        self.audit_logs.append(log_entry)
        
        # Giới hạn kích thước log
        if len(self.audit_logs) > 100000:
            self.audit_logs = self.audit_logs[-50000:]
    
    def log_security_event(self, event_type: str, details: Dict):
        """Ghi log sự kiện bảo mật"""
        event = {
            'id': str(uuid.uuid4()),
            'timestamp': datetime.utcnow().isoformat(),
            'type': event_type,
            'severity': self._calculate_severity(event_type),
            'details': details,
            'hash': self._calculate_hash(details)
        }
        
        self.security_events.append(event)
        
        # Alert nếu severity cao
        if event['severity'] in ['HIGH', 'CRITICAL']:
            self._send_security_alert(event)
    
    def log_compliance_event(self, event_type: str, user_id: str, details: Dict):
        """Ghi log tuân thủ quy định"""
        event = {
            'id': str(uuid.uuid4()),
            'timestamp': datetime.utcnow().isoformat(),
            'type': event_type,
            'user_id': user_id,
            'details': details,
            'regulation': details.get('regulation', 'GENERAL'),
            'hash': self._calculate_hash(details)
        }
        
        self.compliance_logs.append(event)
    
    def _calculate_hash(self, data: Dict) -> str:
        """Tính hash cho log entry để đảm bảo tính toàn vẹn"""
        data_str = json.dumps(data, sort_keys=True)
        return hashlib.sha256(data_str.encode()).hexdigest()
    
    def _calculate_severity(self, event_type: str) -> str:
        """Xác định mức độ nghiêm trọng của sự kiện"""
        severity_map = {
            'UNAUTHORIZED_ACCESS': 'HIGH',
            'BRUTE_FORCE_ATTEMPT': 'HIGH',
            'DATA_BREACH': 'CRITICAL',
            'SUSPICIOUS_TRANSACTION': 'MEDIUM',
            'LOGIN_FAILURE': 'LOW',
            'RATE_LIMIT_EXCEEDED': 'MEDIUM'
        }
        return severity_map.get(event_type, 'LOW')
    
    def _send_security_alert(self, event: Dict):
        """Gửi cảnh báo bảo mật (mô phỏng)"""
        print(f"[SECURITY ALERT] {event['type']} - Severity: {event['severity']}")
        print(f"Details: {event['details']}")
    
    def generate_audit_report(self, start_date: datetime, end_date: datetime) -> Dict:
        """Tạo báo cáo kiểm toán"""
        filtered_logs = [
            log for log in self.audit_logs
            if start_date <= datetime.fromisoformat(log['timestamp']) <= end_date
        ]
        
        report = {
            'period': {
                'start': start_date.isoformat(),
                'end': end_date.isoformat()
            },
            'total_transactions': len(filtered_logs),
            'total_amount': sum(log.get('amount', 0) for log in filtered_logs),
            'security_events': len([
                event for event in self.security_events
                if start_date <= datetime.fromisoformat(event['timestamp']) <= end_date
            ]),
            'high_risk_transactions': len([
                log for log in filtered_logs
                if log.get('risk_score', 0) > 0.7
            ])
        }
        
        return report


In [6]:
! git add . 
! git commit -m "update"
! git push

[master_main 6db198e] update
 1 file changed, 687 insertions(+), 1 deletion(-)


To https://github.com/Trinh-Quoc-Trong/bank_system_oop_python.git
   e6bffe9..6db198e  master_main -> master_main
